# ATESI-SBEC — Reproduction notebook

**Auditable Thermal Energy Storage Intelligence for Smart-Building Energy Prediction and Carbon-Aware Optimization**

This notebook reproduces every quantity, table and result figure of the article from the public
smart-building energy dataset (Kaggle: `waqask/building-energy-dataset`, five annual CSV files,
2016–2021, 3,139,188 one-minute records). Stages 1–9 build the pipeline (data, causal features,
forecasting benchmark, dispatch, recourse, explanation, figures); Stages 10–16 contain the
verification and extended analyses.

**Data.** Download the five CSV files from Kaggle and place them in `{WORK}/data/`. `WORK` defaults
to `/content/atesi` (Colab local disk) and can be changed through the `ATESI_WORK` environment
variable. Intermediate files are written to `{WORK}/pipe/`, figures to `{WORK}/figs/`, and the
JSON results of Stages 10–16 to `{WORK}/results/` (copied to `MyDrive/Outputs/ATESI_SBEC/` when
Google Drive is mounted).

**Software.** LightGBM 4.7.0, XGBoost 3.4.1, scikit-learn 1.9.1, PuLP 2.9.0 (CBC), dice-ml,
statsmodels and PyTorch (CPU). Numerical results of the tree ensembles depend on library versions.

In [ ]:
# Software versions used for all reported results
# !pip install -q lightgbm==4.7.0 xgboost==3.4.1 scikit-learn==1.9.1 pulp==2.9.0 dice-ml statsmodels torch
import lightgbm, xgboost, sklearn, pulp
print('lightgbm', lightgbm.__version__, '| xgboost', xgboost.__version__, '| scikit-learn', sklearn.__version__, '| pulp', pulp.__version__)

In [ ]:
# ---------------------------------------------------------------
# Configuration. Set WORK to the directory holding data/ and pipe/.
# ---------------------------------------------------------------
import os, sys, json, re, time, warnings
warnings.filterwarnings('ignore')

WORK = os.environ.get('ATESI_WORK', '/content/atesi')
os.makedirs(WORK + '/data', exist_ok=True)
os.makedirs(WORK + '/pipe', exist_ok=True)
os.makedirs(WORK + '/figs', exist_ok=True)
print('WORK =', WORK)
print('raw files found:', len(os.listdir(WORK + '/data')))

## Stage 1 — Data ingestion and resampling

*Section 3.1.* Loads the five annual files, de-duplicates and sorts on the timestamp, resamples by
interval mean to the 15-minute control resolution, fills gaps of at most four intervals by linear
interpolation (50 intervals) and drops longer gaps (1,028 intervals).

In [ ]:
import pandas as pd, numpy as np, glob, os
RAW = sorted(glob.glob(WORK+'/data/*.csv'))
STEP = '15min'
def load(paths):
    fr=[]
    for p in paths:
        d=pd.read_csv(p)
        d['Time']=pd.to_datetime(d['Time'],dayfirst=True,errors='coerce')
        fr.append(d)
    df=pd.concat(fr,ignore_index=True).dropna(subset=['Time'])
    df=df.drop_duplicates(subset=['Time']).sort_values('Time').set_index('Time')
    return df
df=load(RAW)
print('raw rows',len(df),'span',df.index.min(),'->',df.index.max())
r=df.resample(STEP).mean()
print('resampled',len(r),'nan rows',int(r.isna().any(axis=1).sum()))
r=r.interpolate(limit=4).dropna()
r.to_csv(WORK+'/pipe/resampled.csv')
print('kept',len(r)); print(r.describe().T.round(3).to_string())

## Stage 2 — Causal feature contract and the same-data benchmark*Manuscript Sections 3.2-3.4 and 4.1.* `build(h, causal=True)` is the leakage contract: everyfeature is a lag, a history-only rolling statistic, or a calendar encoding evaluated at the forecastissue time. Setting `causal=False` reproduces the same-interval construction used in the ablation,so the two rows of Table 7 come from one function with one flag.Trains six estimator families plus persistence and seasonal-naive references on one identicalfeature matrix, one chronological 80/20 split and one target, at the 15-minute and one-hourhorizons. Produces **Tables 4 and 5** and the ablation of **Table 7**.

In [ ]:
import pandas as pd, numpy as np, json, warnings, time
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

df=pd.read_csv(WORK+'/pipe/resampled.csv',index_col=0,parse_dates=True)
TGT='Power[kW]'; EXO=[c for c in df.columns if c!=TGT]
SPD=96  # steps per day at 15min

def cal(idx):
    o=pd.DataFrame(index=idx)
    h=idx.hour+idx.minute/60
    o['hour_sin']=np.sin(2*np.pi*h/24); o['hour_cos']=np.cos(2*np.pi*h/24)
    o['dow_sin']=np.sin(2*np.pi*idx.dayofweek/7); o['dow_cos']=np.cos(2*np.pi*idx.dayofweek/7)
    o['moy_sin']=np.sin(2*np.pi*idx.month/12); o['moy_cos']=np.cos(2*np.pi*idx.month/12)
    o['is_weekend']=(idx.dayofweek>=5).astype(int)
    return o

def build(h, causal=True):
    y=df[TGT]
    F=pd.DataFrame(index=df.index)
    if causal:
        for L in [0,1,2,3,4,8,SPD,SPD*7]: F[f'y_lag{L}']=y.shift(L)
        F['y_rm4']=y.rolling(4).mean(); F['y_rs4']=y.rolling(4).std()
        F['y_rm96']=y.rolling(SPD).mean(); F['y_rs96']=y.rolling(SPD).std()
        F['y_d1']=y.diff(1); F['y_d96']=y.diff(SPD)
        for c in EXO:
            F[f'{c}_lag0']=df[c]; F[f'{c}_lag96']=df[c].shift(SPD)
    else:
        for c in EXO: F[c]=df[c].shift(-h)   # contemporaneous with target
    tgt=y.shift(-h)
    C=cal(df.index.shift(h,freq='15min'))
    C.index=df.index
    F=pd.concat([F,C],axis=1)
    d=pd.concat([F,tgt.rename('y')],axis=1).dropna()
    import re as _re
    d.columns=[_re.sub(r'[^0-9a-zA-Z_]+','_',str(c)).strip('_') for c in d.columns]
    return d.drop(columns='y'), d['y'], d.index

def ev(yt,yp):
    mae=mean_absolute_error(yt,yp); rmse=np.sqrt(mean_squared_error(yt,yp))
    return dict(R2=r2_score(yt,yp),MAE=mae,RMSE=rmse,
                NMAE=mae/np.abs(yt).mean(),NRMSE=rmse/np.abs(yt).mean())

def split(X,y,frac=.8):
    i=int(len(X)*frac); return X.iloc[:i],X.iloc[i:],y.iloc[:i],y.iloc[i:]

MODELS=lambda:{
 'Ridge (scaled)':make_pipeline(StandardScaler(),Ridge(alpha=10.0)),
 'Lasso (scaled)':make_pipeline(StandardScaler(),Lasso(alpha=0.01)),
 'ElasticNet (scaled)':make_pipeline(StandardScaler(),ElasticNet(alpha=0.01,l1_ratio=.5)),
 'Random Forest':RandomForestRegressor(n_estimators=100,min_samples_leaf=5,max_features=0.5,n_jobs=-1,random_state=42),
 'XGBoost':XGBRegressor(n_estimators=600,learning_rate=.05,max_depth=6,subsample=.8,
                        colsample_bytree=.8,n_jobs=-1,random_state=42,tree_method='hist'),
 'LightGBM':LGBMRegressor(n_estimators=600,learning_rate=.05,num_leaves=63,subsample=.8,
                          colsample_bytree=.8,n_jobs=-1,random_state=42,verbose=-1),
}

out={}
for h,lab in [(1,'15 min'),(4,'1 h')]:
    X,y,idx=build(h,causal=True)
    Xtr,Xte,ytr,yte=split(X,y)
    rows=[]
    # naive baselines
    rows.append(dict(Model='Persistence (y_t)',**ev(yte,Xte['y_lag0'])))
    rows.append(dict(Model='Seasonal naive (y_t-24h)',**ev(yte,Xte['y_lag96'])))
    for n,m in MODELS().items():
        t0=time.time(); m.fit(Xtr,ytr); tt=time.time()-t0
        t1=time.time(); p=m.predict(Xte); it=(time.time()-t1)/len(Xte)*1e3
        rows.append(dict(Model=n,**ev(yte,p),TrainSec=tt,LatencyMs_per_1k=it*1000))
    out[lab]=pd.DataFrame(rows)
    print(f'\n===== HORIZON {lab}  (n_train={len(Xtr)}, n_test={len(Xte)}, p={X.shape[1]}) =====')
    print(out[lab].round(4).to_string(index=False))
    out[lab].to_csv(WORK+f'/pipe/forecast_h{h}.csv',index=False)

# ---------- LEAKAGE ABLATION ----------
print('\n===== LEAKAGE ABLATION (Ridge) =====')
ab=[]
Xc,yc,_=build(0,causal=False)          # contemporaneous, as in original notebook
Xa,Xb,ya,yb=split(Xc,yc)
m=make_pipeline(StandardScaler(),Ridge(alpha=10.)); m.fit(Xa,ya)
ab.append(dict(Setting='Contemporaneous features, chronological split (original)',**ev(yb,m.predict(Xb))))
Xs_tr,Xs_te,ys_tr,ys_te=train_test_split(Xc,yc,test_size=.2,shuffle=True,random_state=42)
m=make_pipeline(StandardScaler(),Ridge(alpha=10.)); m.fit(Xs_tr,ys_tr)
ab.append(dict(Setting='Contemporaneous features, SHUFFLED split',**ev(ys_te,m.predict(Xs_te))))
X1,y1,_=build(1,causal=True); Xa,Xb,ya,yb=split(X1,y1)
m=make_pipeline(StandardScaler(),Ridge(alpha=10.)); m.fit(Xa,ya)
ab.append(dict(Setting='Causal lagged features, chronological split (corrected, h=15min)',**ev(yb,m.predict(Xb))))
Xs_tr,Xs_te,ys_tr,ys_te=train_test_split(X1,y1,test_size=.2,shuffle=True,random_state=42)
m=make_pipeline(StandardScaler(),Ridge(alpha=10.)); m.fit(Xs_tr,ys_tr)
ab.append(dict(Setting='Causal lagged features, SHUFFLED split',**ev(ys_te,m.predict(Xs_te))))
A=pd.DataFrame(ab); print(A.round(4).to_string(index=False))
A.to_csv(WORK+'/pipe/leakage_ablation.csv',index=False)

## Stage 3 — Receding-horizon MILP dispatch

*Sections 3.7 and 3.10.* Storage model and dispatch program: charge and discharge powers with a
binary charge-mode indicator, state-of-charge dynamics with one-way efficiencies and standing loss,
state-of-charge bounds (0.10–0.95), power and ramp limits, a terminal state-of-charge floor and
non-negative grid import. The objective sums energy cost under the time-of-use tariff, the demand
charge, the carbon cost and a throughput penalty. Parameter values are those of **Table 2**. Optional
arguments implement the upper-quantile demand term and the billing-period demand charge of Section 4.7.

In [ ]:
"""Receding-horizon MILP TES dispatch with SOC dynamics, real tariff, time-varying carbon.
Parameters as in Table 2."""
import pandas as pd, numpy as np, pulp, json

DT=0.25                      # h per step (15 min)
CAP=200.0                    # kWh usable thermal store
P_MAX=50.0                   # kW charge/discharge rating
ETA_C, ETA_D = 0.95, 0.95    # one-way efficiencies
LOSS=0.002                   # standing loss per step
SOC_MIN, SOC_MAX = 0.10, 0.95  # state-of-charge bounds (fraction of usable capacity), Eq. 13b
SOC0=0.5*CAP
W_CYC=0.005                  # $/kWh throughput penalty (wear and round-trip value), Eq. 19a
RAMP=25.0                    # kW per step
HORIZON=96                   # 24 h look-ahead
STRIDE=96                    # re-solve daily

# --- tariff: real published TOU structure (Ontario-style, $/kWh) + demand charge ---
TOU={'off':0.076,'mid':0.122,'on':0.158}
DEMAND_CHARGE=12.0           # $/kW on monthly coincident peak
def tou_rate(ts):
    if ts.dayofweek>=5: return TOU['off']
    h=ts.hour
    if 7<=h<11 or 17<=h<19: return TOU['on']
    if 11<=h<17: return TOU['mid']
    return TOU['off']
# --- time-varying marginal carbon intensity (kg/kWh), diurnal + seasonal shape ---
def carbon_intensity(ts):
    base=0.35+0.15*np.sin(2*np.pi*(ts.month-1)/12)
    diurn=1.0+0.35*np.sin(2*np.pi*(ts.hour-6)/24)
    return float(np.clip(base*diurn,0.05,0.85))

def solve_window(load, rates, ci, soc_in, w_carbon, pcap=None, load_peak=None, p_hist=None):
    T=len(load)
    m=pulp.LpProblem('tes',pulp.LpMinimize)
    c=[pulp.LpVariable(f'c{t}',lowBound=0,upBound=P_MAX) for t in range(T)]
    d=[pulp.LpVariable(f'd{t}',lowBound=0,upBound=P_MAX) for t in range(T)]
    u=[pulp.LpVariable(f'u{t}',cat='Binary') for t in range(T)]   # 1=charging
    s=[pulp.LpVariable(f's{t}',lowBound=SOC_MIN*CAP,upBound=SOC_MAX*CAP) for t in range(T)]
    g=[pulp.LpVariable(f'g{t}',lowBound=0) for t in range(T)]              # grid import kW
    pk=pulp.LpVariable('pk',lowBound=0)
    for t in range(T):
        m += c[t] <= P_MAX*u[t]
        m += d[t] <= P_MAX*(1-u[t])
        prev = soc_in if t==0 else s[t-1]
        m += s[t] == (1-LOSS)*prev + ETA_C*c[t]*DT - (d[t]*DT)/ETA_D
        m += g[t] >= load[t] + c[t] - d[t]
        m += pk >= g[t]
        if load_peak is not None: m += pk >= load_peak[t] + c[t] - d[t]   # demand term on upper-quantile load
        if pcap is not None: m += g[t] <= pcap
        if t>0:
            m += c[t]-c[t-1] <= RAMP; m += c[t-1]-c[t] <= RAMP
            m += d[t]-d[t-1] <= RAMP; m += d[t-1]-d[t] <= RAMP
    m += s[T-1] >= soc_in*0.9      # no free terminal depletion
    if p_hist is None:
        cost = pulp.lpSum(g[t]*DT*rates[t] for t in range(T)) + DEMAND_CHARGE*pk/30.0
    else:   # billing-period demand charge: only increments above the peak already incurred are charged
        m += pk >= p_hist
        cost = pulp.lpSum(g[t]*DT*rates[t] for t in range(T)) + DEMAND_CHARGE*(pk - p_hist)
    carb = pulp.lpSum(g[t]*DT*ci[t] for t in range(T))
    cyc = pulp.lpSum((c[t]+d[t])*DT*W_CYC for t in range(T))
    m += cost + w_carbon*carb + cyc
    m.solve(pulp.PULP_CBC_CMD(msg=0,timeLimit=30))
    ok = pulp.LpStatus[m.status]=='Optimal'
    return (ok,[v.value() or 0 for v in c],[v.value() or 0 for v in d],
            [v.value() or 0 for v in s],[v.value() or 0 for v in g],pulp.value(m.objective))

def run(pred, actual, index, w_carbon=0.0, ndays=30, pcap=None, upper=None, monthly_demand=False):
    rates=np.array([tou_rate(t) for t in index]); ci=np.array([carbon_intensity(t) for t in index])
    soc=SOC0; rows=[]; solve_times=[]; p_hist=0.0
    import time
    n=min(len(pred), ndays*STRIDE)
    for a in range(0,n-HORIZON,STRIDE):
        w=slice(a,a+HORIZON)
        t0=time.time()
        ok,c,d,s,g,obj=solve_window(np.maximum(pred[w],0),rates[w],ci[w],soc,w_carbon,pcap,None if upper is None else np.maximum(upper[w],0),p_hist if monthly_demand else None)
        solve_times.append(time.time()-t0)
        if not ok: continue
        for k in range(STRIDE):
            if a+k>=n: break
            rows.append(dict(Time=index[a+k],load_pred=pred[a+k],load_act=actual[a+k],
                             charge=c[k],discharge=d[k],soc=s[k],grid=g[k],
                             rate=rates[a+k],ci=ci[a+k]))
        soc=s[STRIDE-1]
        if monthly_demand:   # realized peak so far in the billing period (metered, not forecast)
            p_hist=max(p_hist,max(max(actual[a+k]+c[k]-d[k],0) for k in range(STRIDE) if a+k<n))
    R=pd.DataFrame(rows).set_index('Time')
    R['grid_act']=np.maximum(R.load_act+R.charge-R.discharge,0)
    R['cost_opt']=R.grid_act*DT*R.rate
    R['cost_base']=np.maximum(R.load_act,0)*DT*R.rate
    R['co2_opt']=R.grid_act*DT*R.ci
    R['co2_base']=np.maximum(R.load_act,0)*DT*R.ci
    summ=dict(w_carbon=w_carbon, pcap=pcap,
      days=len(R)/SPD if (SPD:=96) else 0,
      cost_base=R.cost_base.sum(), cost_opt=R.cost_opt.sum(),
      cost_saving_pct=100*(1-R.cost_opt.sum()/R.cost_base.sum()),
      peak_base=np.maximum(R.load_act,0).max(), peak_opt=R.grid_act.max(),
      peak_reduction_pct=100*(1-R.grid_act.max()/np.maximum(R.load_act,0).max()),
      co2_base=R.co2_base.sum(), co2_opt=R.co2_opt.sum(),
      co2_saving_pct=100*(1-R.co2_opt.sum()/R.co2_base.sum()),
      cycles=R.discharge.sum()*DT/CAP,
      demand_base=DEMAND_CHARGE*np.maximum(R.load_act,0).max(), demand_opt=DEMAND_CHARGE*R.grid_act.max(),
      bill_base=R.cost_base.sum()+DEMAND_CHARGE*np.maximum(R.load_act,0).max(),
      bill_opt=R.cost_opt.sum()+DEMAND_CHARGE*R.grid_act.max(),
      soc_min=R.soc.min(), soc_max=R.soc.max(),
      mean_solve_s=float(np.mean(solve_times)), max_solve_s=float(np.max(solve_times)))
    return R,summ

## Stage 4a — Deployed model (shared state for all later stages)Fits the deployed LightGBM forecaster at the one-hour horizon on the causal feature matrix. Theobjects it leaves in the namespace — `m`, `Xtr`, `Xte`, `ytr`, `yte`, `pred` — are consumed byevery stage below, so this cell must be run before stages 4b through 9.

In [ ]:
import pandas as pd, numpy as np, warnings, json, sys, re, time
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from lightgbm import LGBMRegressor

df=pd.read_csv(WORK+'/pipe/resampled.csv',index_col=0,parse_dates=True)
TGT='Power[kW]'; EXO=[c for c in df.columns if c!=TGT]; SPD=96
def cal(idx):
    o=pd.DataFrame(index=range(len(idx)))
    h=idx.hour+idx.minute/60
    o['hour_sin']=np.asarray(np.sin(2*np.pi*h/24)); o['hour_cos']=np.asarray(np.cos(2*np.pi*h/24))
    o['dow_sin']=np.asarray(np.sin(2*np.pi*idx.dayofweek/7)); o['dow_cos']=np.asarray(np.cos(2*np.pi*idx.dayofweek/7))
    o['moy_sin']=np.asarray(np.sin(2*np.pi*idx.month/12)); o['moy_cos']=np.asarray(np.cos(2*np.pi*idx.month/12))
    o['is_weekend']=np.asarray(idx.dayofweek>=5).astype(int)
    return o
def build(h):
    y=df[TGT]; F=pd.DataFrame(index=df.index)
    for L in [0,1,2,3,4,8,SPD,SPD*7]: F[f'y_lag{L}']=y.shift(L)
    F['y_rm4']=y.rolling(4).mean(); F['y_rs4']=y.rolling(4).std()
    F['y_rm96']=y.rolling(SPD).mean(); F['y_rs96']=y.rolling(SPD).std()
    F['y_d1']=y.diff(1); F['y_d96']=y.diff(SPD)
    for c in EXO: F[f'{c}_lag0']=df[c]; F[f'{c}_lag96']=df[c].shift(SPD)
    C=cal(df.index.shift(h,freq='15min')); C.index=df.index
    F=pd.concat([F,C],axis=1)
    d=pd.concat([F,y.shift(-h).rename('y')],axis=1).dropna()
    d.columns=[re.sub(r'[^0-9a-zA-Z_]+','_',str(c)).strip('_') for c in d.columns]
    return d.drop(columns='y'), d['y']
def ev(a,p):
    mae=mean_absolute_error(a,p); rmse=np.sqrt(mean_squared_error(a,p))
    return dict(R2=r2_score(a,p),MAE=mae,RMSE=rmse,NMAE=mae/np.abs(a).mean(),NRMSE=rmse/np.abs(a).mean())
MK=lambda:LGBMRegressor(n_estimators=600,learning_rate=.05,num_leaves=63,subsample=.8,
                        colsample_bytree=.8,n_jobs=-1,random_state=42,verbose=-1)
H=4
X,y=build(H); i=int(len(X)*.8)
Xtr,Xte,ytr,yte=X.iloc[:i],X.iloc[i:],y.iloc[:i],y.iloc[i:]
m=MK(); m.fit(Xtr,ytr); pred=pd.Series(m.predict(Xte),index=Xte.index)
print('=== LightGBM h=1h holdout ==='); print({k:round(v,4) for k,v in ev(yte,pred).items()})

dispatch_run = run   # MILP solver defined in stage 3

## Stage 4b — Rolling-origin backtest, season-wise and regime-stratified errors, dispatch and the cost-carbon frontier

*Sections 4.2, 4.5 and 4.7.* Ten expanding origins (**Table 8**), season-wise accuracy (**Table 9**),
regime stratification with realized decision error under thresholds fixed on the training window
(**Table 10**), and the 29-day dispatch (2,784 intervals) swept over the carbon weight (**Table 11**).

In [ ]:
# ---- rolling-origin backtest, 10 expanding origins ----
print('\n=== ROLLING-ORIGIN BACKTEST (10 origins, expanding) ===')
rows=[]; n=len(X); start=int(n*.5); fold=(n-start)//10
for k in range(10):
    tr_end=start+k*fold; te_end=tr_end+fold
    mm=MK(); mm.fit(X.iloc[:tr_end],y.iloc[:tr_end])
    p=mm.predict(X.iloc[tr_end:te_end])
    r=ev(y.iloc[tr_end:te_end],p); r['Origin']=k+1
    r['TestStart']=str(X.index[tr_end].date()); rows.append(r)
B=pd.DataFrame(rows)[['Origin','TestStart','R2','MAE','RMSE','NMAE','NRMSE']]
print(B.round(4).to_string(index=False))
print('mean R2 %.4f  sd %.4f  min %.4f'%(B.R2.mean(),B.R2.std(),B.R2.min()))
B.to_csv(WORK+'/pipe/backtest.csv',index=False)

# ---- season-wise ----
print('\n=== SEASON-WISE (holdout) ===')
sm={12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'}
S=[]
for s in ['Winter','Spring','Summer','Fall']:
    msk=pd.Series(yte.index.month.map(sm)==s,index=yte.index)
    if msk.sum()>100: S.append(dict(Season=s,n=int(msk.sum()),**ev(yte[msk],pred[msk])))
S=pd.DataFrame(S); print(S.round(4).to_string(index=False)); S.to_csv(WORK+'/pipe/seasonwise.csv',index=False)

# ---- regime-stratified errors ----
print('\n=== REGIME-STRATIFIED ERRORS ===')
ramp=yte.diff().abs(); thr=ramp.quantile(.90); pk=yte.quantile(.90)
regs={'Peak (top demand decile)':yte>=pk,'High ramp (top decile |dP|)':ramp>=thr,
      'Base load (remainder)':(yte<pk)&(ramp<thr)}
R=[]
for k,msk in regs.items():
    msk=msk.fillna(False)
    R.append(dict(Regime=k,n=int(msk.sum()),**ev(yte[msk],pred[msk])))
R=pd.DataFrame(R); print(R.round(4).to_string(index=False))
# decision error rate
ptr_=m.predict(Xtr); q1,q2=np.quantile(ptr_,.25),np.quantile(ptr_,.75)   # thresholds fixed on the training window
a1,a2=q1,q2   # the decision the realized load would have justified under the same thresholds
print('decision thresholds (kW):',round(q1,3),round(q2,3))
dec=lambda v,lo,hi: np.where(v<=lo,'CHARGE',np.where(v<=hi,'STANDBY','DISCHARGE'))
dp=dec(pred,q1,q2); da=dec(yte,a1,a2)
err=[]
for k,msk in regs.items():
    msk=np.asarray(msk.fillna(False))
    err.append(dict(Regime=k,DecisionErrorPct=100*(dp[msk]!=da[msk]).mean()))
E=pd.DataFrame(err); print(E.round(2).to_string(index=False))
R.merge(E,on='Regime').to_csv(WORK+'/pipe/stratified.csv',index=False)

# ---- MILP dispatch + carbon/cost Pareto ----
print('\n=== RECEDING-HORIZON MILP DISPATCH (30 days of holdout) ===')
idx=pred.index; pv=pred.values; av=yte.values
sums=[]
for w in [0.0,0.05,0.10,0.20]:
    t0=time.time(); Rd,su=dispatch_run(pv,av,idx,w_carbon=w,ndays=30)
    su['wall_s']=time.time()-t0; sums.append(su)
    print(f"  w_carbon={w}: cost -{su['cost_saving_pct']:.2f}%  peak -{su['peak_reduction_pct']:.2f}%  "
          f"CO2 -{su['co2_saving_pct']:.2f}%  cycles {su['cycles']:.2f}  solve {su['mean_solve_s']:.2f}s")
    if w==0.10: Rd.to_csv(WORK+'/pipe/dispatch_schedule.csv')
P=pd.DataFrame(sums); P.to_csv(WORK+'/pipe/pareto.csv',index=False)
print('\n'+P[['w_carbon','cost_base','cost_opt','cost_saving_pct','peak_base','peak_opt',
              'peak_reduction_pct','co2_base','co2_opt','co2_saving_pct','cycles','soc_min','soc_max',
              'mean_solve_s','max_solve_s']].round(3).to_string(index=False))
print('\nDONE')

## Stage 5 — Peak-cap experiment

*Sections 4.7 and 5.* Imposes a hard cap on grid import at 100, 90 and 80 per cent of the baseline
realized peak. The cap is written on predicted import, so it binds in the optimizer and not on the
realized load: the realized peak is unchanged in all three cases.

In [ ]:
dr = run   # MILP solver defined in stage 3
import pandas as pd,numpy as np,sys,warnings,re,time
idx=pred.index; pv=pred.values; av=yte.values
base_peak=float(np.maximum(av[:30*96],0).max())
print('baseline realized peak %.3f kW'%base_peak)
rows=[]
for cap_frac in [None,1.00,0.90,0.80]:
    cap=None if cap_frac is None else base_peak*cap_frac
    R,su=dr(pv,av,idx,w_carbon=0.10,ndays=30,pcap=cap)
    su['cap_frac']='none' if cap_frac is None else cap_frac
    rows.append(su)
    print(f"  cap={su['cap_frac']}: cost {su['cost_saving_pct']:+.2f}%  peak {su['peak_reduction_pct']:+.2f}%  "
          f"CO2 {su['co2_saving_pct']:+.2f}%  cycles {su['cycles']:.2f}")
    if cap_frac==0.90: R.to_csv(WORK+'/pipe/dispatch_cap90.csv')
P=pd.DataFrame(rows)
cols=['cap_frac','cost_base','cost_opt','cost_saving_pct','peak_base','peak_opt','peak_reduction_pct',
      'co2_base','co2_opt','co2_saving_pct','cycles','soc_min','soc_max','mean_solve_s','max_solve_s']
print('\n'+P[cols].round(3).to_string(index=False)); P.to_csv(WORK+'/pipe/peakcap.csv',index=False)
print('DONE')

## Stage 6 — DiCE counterfactual recourse

*Sections 3.11 and 4.10.* Counterfactuals from DiCE (model-agnostic random method, regression mode)
for a 25 per cent demand-reduction request on a low-, a medium- and a peak-demand record. Consuming
sub-meters may only decrease (floor 0), photovoltaic generation may only increase (floor at the
training minimum), the battery and all lag and calendar terms are frozen, and candidates must lie in
the empirical joint operating envelope (Mahalanobis, 99th percentile) and respect the observed
chiller coupling ratios. Ten seeds per record. Produces **Table 14** and the audited decision log of
**Table 13**.

In [ ]:
import pandas as pd, numpy as np, warnings, dice_ml
warnings.filterwarnings('ignore')
# Counterfactual recourse with DiCE (model-agnostic random method), regression mode.
# Actionable set: consuming sub-meters may only decrease (floor 0); PV may only increase generation
# (more negative, floored at the observed training minimum); battery, lags, rolling and calendar terms are frozen.
ACT0 = [c for c in Xte.columns if c.endswith('_lag0') and not c.startswith('y_')]
CONSUME = [c for c in ACT0 if c.startswith(('HVAC', 'Chiller', 'Humidifier', 'HV_light'))]
PVC = [c for c in ACT0 if c.startswith('PV')]; ACT = CONSUME + PVC
PVFLOOR = {c: float(Xtr[c].min()) for c in PVC}
cols = list(Xte.columns); ai = [cols.index(c) for c in ACT]
# joint operating envelope (Mahalanobis, 99th percentile of training) and coupling ratios (1st-99th percentile)
A_ = Xtr[ACT].values; muA = A_.mean(0); CiA = np.linalg.pinv(np.cov(A_.T))
mdist = lambda V: np.sqrt(np.einsum('ij,jk,ik->i', V - muA, CiA, V - muA))
MD99 = float(np.quantile(mdist(A_), .99))
ch_ = [c for c in ACT if c.startswith('Chiller')][0]; hu_ = [c for c in ACT if c.startswith('Humidifier')][0]; hv_ = [c for c in ACT if c.startswith('HVAC')][0]
EPS = 1e-3
r1 = (Xtr[hu_] + EPS) / (Xtr[ch_] + EPS); r2 = (Xtr[hv_] + EPS) / (Xtr[ch_] + EPS)
B1 = (r1.quantile(.01), r1.quantile(.99)); B2 = (r2.quantile(.01), r2.quantile(.99))
def admissible(V):
    ok = mdist(V[ACT].values) <= MD99
    a = (V[hu_] + EPS) / (V[ch_] + EPS); b = (V[hv_] + EPS) / (V[ch_] + EPS)
    return ok & (a >= B1[0]).values & (a <= B1[1]).values & (b >= B2[0]).values & (b <= B2[1]).values
print('envelope MD99 = %.3f; ratio bounds hum/chiller [%.4f, %.2f], HVAC/chiller [%.3f, %.2f]' % (MD99, *B1, *B2), flush=True)

samp = Xtr.sample(20000, random_state=0).copy(); samp['y'] = m.predict(samp[cols])
dd = dice_ml.Data(dataframe=samp, continuous_features=cols, outcome_name='y')
mm_ = dice_ml.Model(model=m, backend='sklearn', model_type='regressor')
EXP = dice_ml.Dice(dd, mm_, method='random')

def recourse(row, target, seed, tol=0.15, n_cf=60):
    q = row.to_frame().T.astype(float)
    pr = {c: [0.0, float(row[c])] for c in CONSUME}
    pr.update({c: [min(PVFLOOR[c], float(row[c])), float(row[c])] for c in PVC})
    try:
        r = EXP.generate_counterfactuals(q, total_CFs=n_cf, desired_range=[target - tol, target + tol],
                                         features_to_vary=ACT, permitted_range=pr, random_seed=seed, verbose=False)
        C = r.cf_examples_list[0].final_cfs_df
    except Exception as e:
        print('    DiCE returned no counterfactual:', type(e).__name__, flush=True); C = None
    span = np.array([max(float(row[c]) - pr[c][0], 1e-9) for c in ACT])
    if C is None or len(C) == 0:
        return None, None, False, 0
    C = C[cols].astype(float); P = m.predict(C)
    keep = admissible(C) & (np.abs(P - target) <= tol)
    n_adm = int(keep.sum())
    if n_adm == 0:
        return None, None, False, 0
    C = C[keep]; P = P[keep]
    cost = (np.abs(C[ACT].values - row[ACT].values.astype(float)) / span).sum(1)
    k = int(np.argmin(cost)); return C.iloc[k], float(P[k]), True, n_adm

def bound_search(row, target, seed, n=40000):
    """Closest attainable admissible point when no counterfactual reaches the request (exhaustive sampling)."""
    rng = np.random.default_rng(seed); base = row.values.astype(float)
    L = np.array([0.0 if c in CONSUME else min(PVFLOOR[c], base[cols.index(c)]) for c in ACT]); H_ = base[ai]
    S = np.tile(base, (n, 1)); S[:, ai] = rng.uniform(np.minimum(L, H_), H_, size=(n, len(ACT)))
    Sd = pd.DataFrame(S, columns=cols); ok = admissible(Sd); p = m.predict(Sd)
    k = np.where(ok)[0][np.argmin(np.abs(p[ok] - target))]; return Sd.iloc[k], float(p[k])

recs = {}
q = pred.quantile([.10, .50, .95])
for lab, qv in [('Low demand', q.loc[.10]), ('Medium demand', q.loc[.50]), ('Peak demand', q.loc[.95])]:
    i = int(np.argmin(np.abs(pred.values - qv))); recs[lab] = (Xte.index[i], Xte.iloc[i])
rows = []; stab = {}
for lab, (ts, row) in recs.items():
    p0 = float(m.predict(row.to_frame().T.astype(float))[0]); tgt = 0.75 * p0
    res = []
    for seed in range(10):
        v, p1, feas, nadm = recourse(row, tgt, seed)
        if not feas: v, p1 = bound_search(row, tgt, seed)
        res.append((v, p1, feas, nadm))
        print(f'  {lab} seed {seed}: feasible={feas} achieved={p1:.3f} kW (admissible CFs {nadm})', flush=True)
    feas_all = [r[2] for r in res]
    ach = np.array([r[1] for r in res]); D = np.array([r[0][ACT].values.astype(float) - row[ACT].values.astype(float) for r in res])
    stab[lab] = dict(feasible_frac=float(np.mean(feas_all)), achieved_mean=float(ach.mean()), achieved_sd=float(ach.std()),
                     physical_signs=bool((D <= 1e-9).all()), delta_sd_max=float(D.std(0).max()))
    v, p1 = res[0][0], res[0][1]
    d = dict(Record=lab, Timestamp=str(ts), Original_kW=round(p0, 3), Target_kW=round(tgt, 3), Achieved_kW=round(p1, 3), Feasible=res[0][2])
    for c in ACT: d[c.replace('_lag0', '')] = f"{float(row[c]):.3f} -> {float(v[c]):.3f}"
    rows.append(d)
CF = pd.DataFrame(rows); print('\n=== DiCE COUNTERFACTUALS (25% demand reduction, envelope- and coupling-constrained) ===')
print(CF.to_string(index=False)); CF.to_csv(WORK + '/pipe/counterfactuals.csv', index=False)
print('stability across 10 seeds:', stab)
import json; json.dump(dict(stability=stab, MD99=MD99, B1=list(map(float, B1)), B2=list(map(float, B2))), open(WORK + '/pipe/cf_stability.json', 'w'), indent=1)

# audited decision log (Table 13): three-hourly samples of the opening day of the dispatch window
S_ = pd.read_csv(WORK + '/pipe/dispatch_schedule.csv', index_col=0, parse_dates=True)
S_['action'] = np.where(S_.charge > 0.1, 'Charge', np.where(S_.discharge > 0.1, 'Discharge', 'Standby'))
L_ = S_.iloc[:96:12].copy(); L_['soc_pct'] = 100 * L_.soc / CAP; L_['energy_kWh'] = L_.grid_act * DT
L_['cost'] = L_.energy_kWh * L_.rate; L_['co2'] = L_.energy_kWh * L_.ci
L_.to_csv(WORK + '/pipe/decision_log.csv'); print(L_[['load_act','load_pred','action','charge','discharge','soc_pct','grid_act','rate','energy_kWh','cost','ci','co2']].round(3).to_string())

## Stage 7 — Diebold-Mariano tests and storage-parameter sensitivity

*Section 4.1 (Table 6) and Section 4.7 (Table 12).* Diebold-Mariano statistics with a Newey-West
correction under squared- and absolute-error loss, and the one-factor sensitivity sweep over usable
capacity, power rating, one-way efficiency, standing loss and ramp limit.

In [ ]:
drun = run   # MILP solver defined in stage 3
import pandas as pd,numpy as np,warnings,re,sys,json
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from scipy import stats

# ---------------- 1. Diebold-Mariano tests ----------------
P={'LightGBM':pred.values}
P['Persistence']=Xte['y_lag0'].values
P['Seasonal naive']=Xte['y_lag96'].values
for n,mk in [('Ridge (scaled)',make_pipeline(StandardScaler(),Ridge(alpha=10.))),
             ('XGBoost',XGBRegressor(n_estimators=600,learning_rate=.05,max_depth=6,subsample=.8,
                        colsample_bytree=.8,n_jobs=-1,random_state=42,tree_method='hist')),
             ('Random forest',RandomForestRegressor(n_estimators=100,min_samples_leaf=5,max_features=0.5,
                        n_jobs=-1,random_state=42))]:
    mk.fit(Xtr,ytr); P[n]=mk.predict(Xte)
y=yte.values
def dm(e1,e2,h=4,power=2):
    """Diebold-Mariano with Newey-West HAC (lag h-1); positive stat => model 1 worse."""
    d=np.abs(e1)**power-np.abs(e2)**power
    T=len(d); dbar=d.mean(); L=h-1
    g0=np.mean((d-dbar)**2)
    var=g0
    for l in range(1,L+1):
        gl=np.mean((d[l:]-dbar)*(d[:-l]-dbar))
        var+=2*(1-l/(L+1))*gl
    se=np.sqrt(var/T)
    stat=dbar/se
    p=2*(1-stats.t.cdf(abs(stat),T-1))
    return stat,p,dbar
rows=[]
base=y-P['LightGBM']
for n,p_ in P.items():
    if n=='LightGBM': continue
    e_other=y-p_
    s,pv,dbar=dm(e_other,base,h=4,power=2)     # >0 => other worse than LightGBM
    s1,pv1,_=dm(e_other,base,h=4,power=1)
    rows.append(dict(Comparison=f'LightGBM vs {n}',DM_MSE=s,p_MSE=pv,DM_MAE=s1,p_MAE=pv1,
                     dMSE=dbar))
DM=pd.DataFrame(rows)
print('=== DIEBOLD-MARIANO (h = 1 h, HAC lag 3; positive statistic favours LightGBM) ===')
print(DM.round(4).to_string(index=False))
DM.to_csv(WORK+'/pipe/dm_tests.csv',index=False)

# ---------------- 2. Backtest confidence interval ----------------
B=pd.read_csv(WORK+'/pipe/backtest.csv')
n=len(B); mR=B.R2.mean(); sR=B.R2.std(ddof=1)
tcrit=stats.t.ppf(0.975,n-1); half=tcrit*sR/np.sqrt(n)
print(f'\n=== BACKTEST R2: mean {mR:.4f}, sd {sR:.4f}, 95% CI [{mR-half:.4f}, {mR+half:.4f}] (n={n}) ===')
json.dump(dict(mean=mR,sd=sR,lo=mR-half,hi=mR+half,n=n),open(WORK+'/pipe/backtest_ci.json','w'))

# ---------------- 3. Storage-parameter sensitivity ----------------
idx=pred.index; pv=pred.values; av=yte.values
_G=globals()
BASE=dict(CAP=200.,P_MAX=50.,ETA_C=.95,ETA_D=.95,LOSS=.002,RAMP=25.)
def setp(**kw):
    for k,v in BASE.items(): _G[k]=v
    for k,v in kw.items(): _G[k]=v
    _G['SOC0']=0.5*_G['CAP']
grid=[('baseline',{})]
for v in [100.,400.]: grid.append((f'capacity = {v:.0f} kWh',dict(CAP=v)))
for v in [25.,100.]: grid.append((f'power rating = {v:.0f} kW',dict(P_MAX=v)))
for v in [.90,.98]: grid.append((f'one-way efficiency = {v:.2f}',dict(ETA_C=v,ETA_D=v)))
for v in [.001,.005]: grid.append((f'standing loss = {v:.3f}/interval',dict(LOSS=v)))
for v in [12.5,50.]: grid.append((f'ramp limit = {v:.1f} kW/interval',dict(RAMP=v)))
out=[]
for lab,kw in grid:
    setp(**kw)
    R,su=drun(pv,av,idx,w_carbon=0.10,ndays=30)
    out.append(dict(Configuration=lab,cost_pct=su['cost_saving_pct'],co2_pct=su['co2_saving_pct'],
                    cycles=su['cycles'],soc_max=su['soc_max'],solve=su['mean_solve_s'],
                    bill_pct=100*(1-su['bill_opt']/su['bill_base']),peak_opt=su['peak_opt']))
    print(f"  {lab:34s} cost {su['cost_saving_pct']:6.2f}%   CO2 {su['co2_saving_pct']:5.2f}%   cycles {su['cycles']:5.2f}",flush=True)
setp()
S=pd.DataFrame(out); S.to_csv(WORK+'/pipe/sensitivity.csv',index=False)
print('\n=== SENSITIVITY ==='); print(S.round(3).to_string(index=False))
print(f"cost range {S.cost_pct.min():.2f} to {S.cost_pct.max():.2f} %;  CO2 range {S.co2_pct.min():.2f} to {S.co2_pct.max():.2f} %")
print('DONE')


## Stage 8 — LIME local attributions*Manuscript Section 4.4, Figures 4-6.* Five thousand perturbations per explained instance instandardized feature space, weighted by an exponential proximity kernel, with a weighted ridgesurrogate fitted to the forecaster's responses. The three intervals explained are the same low,mid-range and peak intervals for which recourse is generated in Section 4.10, so explanation andrecommended action can be read together. Surrogate fidelity is reported per panel.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd,numpy as np,warnings,re,sys
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8,'axes.grid':True,'grid.alpha':.3,'grid.linestyle':'--',
                     'axes.titlesize':9,'axes.titleweight':'bold'})
from sklearn.linear_model import Ridge
M=WORK+'/figs/'
rng=np.random.default_rng(0)
mu=Xtr.mean().values; sd=Xtr.std().values.copy(); sd[sd==0]=1.0
cols=list(Xtr.columns)
def pretty(c):
    c=c.replace('_kW_','').replace('_lag0',' (t)').replace('_lag96',' (t-24h)')
    c=c.replace('y_lag0','load (t)').replace('y_lag','load (t-').replace('y_rm','roll mean ').replace('y_rs','roll sd ')
    c=c.replace('y_d1','Δload 1').replace('y_d96','Δload 24h').replace('_',' ')
    return c[:34]
def lime_explain(x0, n=5000, k=8, width=None):
    """Local surrogate: sample around x0 in z-space, weight by exponential kernel, fit weighted ridge."""
    z0=(x0-mu)/sd
    Z=rng.normal(0,1,size=(n,len(cols)))          # perturbations in standardized space
    Z[0]=z0
    X=Z*sd+mu
    yhat=m.predict(X)
    d=np.sqrt(((Z-z0)**2).sum(1))
    width=width or np.sqrt(len(cols))*0.75
    w=np.exp(-(d**2)/(width**2))
    sur=Ridge(alpha=1.0); sur.fit(Z, yhat, sample_weight=w)
    coef=sur.coef_
    idx=np.argsort(-np.abs(coef))[:k]
    return [(pretty(cols[i]), float(coef[i])) for i in idx], float(m.predict(x0.reshape(1,-1))[0]), sur.score(Z,yhat,sample_weight=w)
targets=[('Low',pred.quantile(.10)),('Medium',pred.quantile(.50)),('Peak',pred.quantile(.95))]
ars=[1.728,1.618,1.728]; names=['image4.png','image5.png','image6.png']
info=[]
for (lab,q),ar,nm in zip(targets,ars,names):
    i=int(np.argmin(np.abs(pred.values-q)))
    x0=Xte.iloc[i].values.astype(float)
    ex,p0,r2=lime_explain(x0)
    ts=Xte.index[i]
    f,a=plt.subplots()
    feats=[e[0] for e in ex][::-1]; vals=[e[1] for e in ex][::-1]
    a.barh(feats,vals,color=['#c53030' if v>0 else '#2b6cb0' for v in vals])
    a.axvline(0,color='k',lw=.8)
    a.set_title(f'Local attribution, {lab.lower()}-demand interval {ts:%Y-%m-%d %H:%M}\nforecast = {p0:.2f} kW  (local surrogate R² = {r2:.3f})')
    a.set_xlabel('Contribution to forecast (kW per standard deviation of feature)')
    a.tick_params(axis='y',labelsize=7)
    w=7.5; f.set_size_inches(w,w/ar); f.tight_layout(); f.savefig(M+nm,dpi=170); plt.close(f)
    info.append((lab,str(ts),round(p0,2),round(r2,3),[(n_,round(v,3)) for n_,v in ex[:4]]))
    print('wrote',nm,lab,ts,round(p0,2),'surrogate R2',round(r2,3),flush=True)
for r in info: print(r)

## Stage 9 — Result figures

*Figures 2, 3 and 7–16.* Regenerates the multi-panel result figures from the dispatch schedule and the
held-out predictions (Figure 1 is the architecture diagram). Figures are written to `{WORK}/figs/`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import pandas as pd,numpy as np,warnings,re,sys
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8,'axes.grid':True,'grid.alpha':.3,'grid.linestyle':'--',
                     'axes.titlesize':9,'axes.titleweight':'bold'})
M=WORK+'/figs/'
res=yte-pred
S=pd.read_csv(WORK+'/pipe/dispatch_schedule.csv',index_col=0,parse_dates=True)
S['action']=np.where(S.charge>0.1,'CHARGE',np.where(S.discharge>0.1,'DISCHARGE','STANDBY'))
DT=0.25; S['energy']=S.grid_act*DT; S['cost']=S.energy*S.rate; S['co2']=S.energy*S.ci
S['base_e']=np.maximum(S.load_act,0)*DT; S['base_cost']=S.base_e*S.rate; S['base_co2']=S.base_e*S.ci
S['soc_pct']=100*S.soc/CAP
BK=pd.read_csv(WORK+'/pipe/backtest.csv'); ST=pd.read_csv(WORK+'/pipe/stratified.csv')
SE=pd.read_csv(WORK+'/pipe/seasonwise.csv'); PA=pd.read_csv(WORK+'/pipe/pareto.csv')
C1,C2,C3='#2b6cb0','#c05621','#2f855a'
def save(fig,name,ar):
    w=9.0; fig.set_size_inches(w,w/ar); fig.tight_layout(); fig.savefig(M+name,dpi=160); plt.close(fig); print('wrote',name,flush=True)
reg=[r.split(' (')[0] for r in ST.Regime]

f,a=plt.subplots(1,3)
n=800; a[0].plot(yte.values[-n:],lw=.7,color='#444',label='Actual'); a[0].plot(pred.values[-n:],lw=.7,color=C1,label='Forecast')
a[0].set_title('(A) One-hour-ahead forecast vs actual'); a[0].set_xlabel('Interval'); a[0].set_ylabel('Net demand (kW)'); a[0].legend(fontsize=6)
a[1].hist(res,bins=80,color=C1); a[1].axvline(0,color='k',lw=.8); a[1].set_title('(B) Residual distribution'); a[1].set_xlabel('Residual (kW)'); a[1].set_ylabel('Frequency')
a[2].scatter(pred,res,s=.5,alpha=.15,color=C2); a[2].axhline(0,color='k',lw=.8); a[2].set_title('(C) Residual vs forecast'); a[2].set_xlabel('Forecast (kW)'); a[2].set_ylabel('Residual (kW)')
save(f,'image2.png',3.611)

f,a=plt.subplots(2,2)
a[0,0].scatter(yte,pred,s=.5,alpha=.15,color=C1); lim=[min(yte.min(),pred.min()),max(yte.max(),pred.max())]
a[0,0].plot(lim,lim,'k--',lw=.8); a[0,0].set_title('(A) Parity plot, held-out window'); a[0,0].set_xlabel('Actual (kW)'); a[0,0].set_ylabel('Forecast (kW)')
a[0,1].bar(range(len(BK)),BK.R2,color=C1); a[0,1].axhline(BK.R2.mean(),color=C2,ls='--',lw=1,label=f'mean {BK.R2.mean():.3f}')
a[0,1].set_xticks(range(len(BK))); a[0,1].set_xticklabels(BK.Origin); a[0,1].legend(fontsize=6); a[0,1].set_title('(B) Rolling-origin backtest'); a[0,1].set_xlabel('Origin'); a[0,1].set_ylabel('R²')
a[1,0].bar(reg,ST.MAE,color=C3); a[1,0].set_title('(C) MAE by operating regime'); a[1,0].set_ylabel('MAE (kW)'); a[1,0].tick_params(axis='x',labelsize=6)
a[1,1].bar(SE.Season,SE.R2,color=C1); a[1,1].set_title('(D) Season-wise R²'); a[1,1].set_ylabel('R²')
save(f,'image3.png',1.817)

ptr_=m.predict(Xtr); q1,q2=np.quantile(ptr_,.25),np.quantile(ptr_,.75)
cls=np.where(pred<=q1,'Low',np.where(pred<=q2,'Medium','Peak'))
f,a=plt.subplots(1,3)
a[0].boxplot([pred[cls==k] for k in ['Low','Medium','Peak']],tick_labels=['Low','Medium','Peak'])
a[0].set_title('(A) Forecast by demand class'); a[0].set_ylabel('Forecast (kW)')
for k,c in zip(['Low','Medium','Peak'],[C3,C2,'#c53030']): a[1].scatter(yte[cls==k],pred[cls==k],s=.5,alpha=.2,color=c,label=k)
a[1].legend(fontsize=6,markerscale=8); a[1].set_title('(B) Actual vs forecast by class'); a[1].set_xlabel('Actual (kW)'); a[1].set_ylabel('Forecast (kW)')
vc=pd.Series(cls).value_counts()[['Low','Medium','Peak']]
a[2].bar(vc.index,vc.values,color=[C3,C2,'#c53030']); a[2].set_title('(C) Class frequency'); a[2].set_ylabel('Intervals')
save(f,'image7.png',3.161)

f,a=plt.subplots(2,2); d1=S.iloc[:96]
a[0,0].plot(d1.index,d1.load_act,color='#444',lw=.9,label='Load'); a[0,0].plot(d1.index,d1.grid_act,color=C1,lw=.9,label='Grid import')
a[0,0].legend(fontsize=6); a[0,0].set_title('(A) Load and grid import, one day'); a[0,0].set_ylabel('kW'); a[0,0].tick_params(axis='x',rotation=30,labelsize=6)
a[0,1].plot(d1.index,d1.soc_pct,color=C3,lw=1); a[0,1].set_title('(B) State-of-charge trajectory'); a[0,1].set_ylabel('SoC (%)'); a[0,1].tick_params(axis='x',rotation=30,labelsize=6)
a[1,0].bar(d1.index,d1.charge,width=.01,color=C3,label='Charge'); a[1,0].bar(d1.index,-d1.discharge,width=.01,color='#c53030',label='Discharge')
a[1,0].legend(fontsize=6); a[1,0].set_title('(C) Charge and discharge power'); a[1,0].set_ylabel('kW'); a[1,0].tick_params(axis='x',rotation=30,labelsize=6)
ac=S.action.value_counts(); a[1,1].bar(ac.index,ac.values,color=[C3,'#c53030',C2][:len(ac)])
a[1,1].set_title('(D) Action frequency, 30 days'); a[1,1].set_ylabel('Intervals')
save(f,'image8.png',1.41)

f,a=plt.subplots(2,2)
a[0,0].hist(yte,bins=60,color=C1,alpha=.7,label='Actual'); a[0,0].hist(pred,bins=60,color=C2,alpha=.6,label='Forecast')
a[0,0].legend(fontsize=6); a[0,0].set_title('(A) Demand distribution'); a[0,0].set_xlabel('kW')
a[0,1].plot(np.sort(np.abs(res)),np.linspace(0,1,len(res)),color=C1,lw=1); a[0,1].set_title('(B) Absolute error CDF'); a[0,1].set_xlabel('|error| (kW)'); a[0,1].set_ylabel('F')
a[1,0].bar(reg,ST['DecisionErrorPct'],color=C2); a[1,0].set_title('(C) Decision error by regime'); a[1,0].set_ylabel('%'); a[1,0].tick_params(axis='x',labelsize=6)
hb=pred.groupby(pred.index.hour).mean(); ha=yte.groupby(yte.index.hour).mean()
a[1,1].plot(ha.index,ha.values,color='#444',lw=1,label='Actual'); a[1,1].plot(hb.index,hb.values,color=C1,lw=1,label='Forecast')
a[1,1].legend(fontsize=6); a[1,1].set_title('(D) Mean diurnal profile'); a[1,1].set_xlabel('Hour'); a[1,1].set_ylabel('kW')
save(f,'image9.png',1.415)

f,a=plt.subplots(2,3)
a[0,0].hist(S.cost,bins=50,color=C1); a[0,0].set_title('(A) Interval cost distribution'); a[0,0].set_xlabel('$')
cc=S.groupby(S.rate).cost.sum(); a[0,1].bar([f'{r:.3f}' for r in cc.index],cc.values,color=C2)
a[0,1].set_title('(B) Cost by tariff tier'); a[0,1].set_ylabel('$')
a[0,2].scatter(S.grid_act,S.cost,s=.6,alpha=.3,color=C1); a[0,2].set_title('(C) Grid import vs cost'); a[0,2].set_xlabel('kW'); a[0,2].set_ylabel('$')
a[1,0].plot(S.index,S.base_cost.cumsum(),color='#444',lw=1,label='Unmanaged'); a[1,0].plot(S.index,S.cost.cumsum(),color=C3,lw=1,label='Dispatched')
a[1,0].legend(fontsize=6); a[1,0].set_title('(D) Cumulative cost'); a[1,0].set_ylabel('$'); a[1,0].tick_params(axis='x',rotation=30,labelsize=6)
a[1,1].bar(['Unmanaged','Dispatched'],[S.base_cost.sum(),S.cost.sum()],color=['#444',C3]); a[1,1].set_title('(E) Total 30-day cost'); a[1,1].set_ylabel('$')
a[1,2].plot(PA.cost_saving_pct,PA.co2_saving_pct,'o-',color=C2)
for _,r in PA.iterrows(): a[1,2].annotate(f"w={r.w_carbon:g}",(r.cost_saving_pct,r.co2_saving_pct),fontsize=6)
a[1,2].set_title('(F) Cost-carbon frontier'); a[1,2].set_xlabel('Cost reduction (%)'); a[1,2].set_ylabel('CO₂ reduction (%)')
save(f,'image10.png',1.818)

f,a=plt.subplots(2,2)
a[0,0].plot(PA.w_carbon,PA.cost_saving_pct,'o-',color=C1); a[0,0].set_title('(A) Cost reduction vs carbon weight'); a[0,0].set_xlabel('Carbon weight'); a[0,0].set_ylabel('%')
a[0,1].plot(PA.w_carbon,PA.co2_saving_pct,'o-',color=C3); a[0,1].set_title('(B) CO₂ reduction vs carbon weight'); a[0,1].set_xlabel('Carbon weight'); a[0,1].set_ylabel('%')
a[1,0].plot(PA.w_carbon,PA.cycles,'o-',color=C2); a[1,0].set_title('(C) Cycling vs carbon weight'); a[1,0].set_xlabel('Carbon weight'); a[1,0].set_ylabel('Equivalent full cycles')
a[1,1].bar(['mean','max'],[PA.mean_solve_s.mean(),PA.max_solve_s.max()],color=C1); a[1,1].set_title('(D) MILP solution time per step'); a[1,1].set_ylabel('s')
save(f,'image11.png',1.409)

f,a=plt.subplots(2,2)
a[0,0].hist(S.base_co2,bins=50,color='#444',alpha=.7,label='Unmanaged'); a[0,0].hist(S.co2,bins=50,color=C3,alpha=.7,label='Dispatched')
a[0,0].legend(fontsize=6); a[0,0].set_title('(A) Interval emissions'); a[0,0].set_xlabel('kg CO₂')
ci=S.groupby(S.index.hour).ci.mean(); a[0,1].plot(ci.index,ci.values,color=C2,lw=1)
a[0,1].set_title('(B) Mean diurnal carbon intensity'); a[0,1].set_xlabel('Hour'); a[0,1].set_ylabel('kg/kWh')
a[1,0].plot(S.index,S.base_co2.cumsum(),color='#444',lw=1,label='Unmanaged'); a[1,0].plot(S.index,S.co2.cumsum(),color=C3,lw=1,label='Dispatched')
a[1,0].legend(fontsize=6); a[1,0].set_title('(C) Cumulative emissions'); a[1,0].set_ylabel('kg CO₂'); a[1,0].tick_params(axis='x',rotation=30,labelsize=6)
a[1,1].bar(['Unmanaged','Dispatched'],[S.base_co2.sum(),S.co2.sum()],color=['#444',C3]); a[1,1].set_title('(D) Total 30-day emissions'); a[1,1].set_ylabel('kg CO₂')
save(f,'image12.png',1.817)

red=100*(1-S.co2.cumsum()/S.base_co2.cumsum())
f,a=plt.subplots(2,2)
a[0,0].plot(S.index,red,color=C3,lw=1); a[0,0].set_title('(A) Cumulative carbon reduction'); a[0,0].set_ylabel('%'); a[0,0].tick_params(axis='x',rotation=30,labelsize=6)
byact=S.groupby('action')[['cost','co2']].sum()
a[0,1].bar(byact.index,byact.cost,color=C1); a[0,1].set_title('(B) Cost by action'); a[0,1].set_ylabel('$')
a[1,0].bar(byact.index,byact.co2,color=C3); a[1,0].set_title('(C) Emissions by action'); a[1,0].set_ylabel('kg CO₂')
a[1,1].scatter(S.ci,S.discharge,s=.6,alpha=.3,color=C2); a[1,1].set_title('(D) Discharge vs carbon intensity'); a[1,1].set_xlabel('kg/kWh'); a[1,1].set_ylabel('Discharge (kW)')
save(f,'image13.png',1.817)

f,a=plt.subplots(2,2)
a[0,0].hist(S.soc_pct,bins=50,color=C3); a[0,0].set_title('(A) State-of-charge distribution'); a[0,0].set_xlabel('%')
a[0,1].scatter(S.rate,S.charge,s=.6,alpha=.3,color=C1); a[0,1].set_title('(B) Charging vs tariff rate'); a[0,1].set_xlabel('$/kWh'); a[0,1].set_ylabel('Charge (kW)')
a[1,0].plot(S.index,S.soc_pct,color=C3,lw=.6); a[1,0].set_title('(C) State-of-charge over 30 days'); a[1,0].set_ylabel('%'); a[1,0].tick_params(axis='x',rotation=30,labelsize=6)
a[1,1].bar(['Cost','CO₂'],[100*(1-S.cost.sum()/S.base_cost.sum()),100*(1-S.co2.sum()/S.base_co2.sum())],color=[C1,C3])
a[1,1].set_title('(D) Realized reductions'); a[1,1].set_ylabel('%')
save(f,'image14.png',1.387)

f,a=plt.subplots(2,2)
a[0,0].bar(ac.index,ac.values,color=[C3,'#c53030',C2][:len(ac)]); a[0,0].set_title('(A) Action distribution'); a[0,0].set_ylabel('Intervals')
a[0,1].scatter(S.cost,S.co2,s=.6,alpha=.3,color=C1); a[0,1].set_title('(B) Cost vs emissions per interval'); a[0,1].set_xlabel('$'); a[0,1].set_ylabel('kg CO₂')
ah=S.groupby([S.index.hour,'action']).size().unstack(fill_value=0)
ah.plot(kind='bar',stacked=True,ax=a[1,0],width=.9,legend=False,color=[C3,'#c53030',C2][:ah.shape[1]])
a[1,0].set_title('(C) Action by hour of day'); a[1,0].set_xlabel('Hour'); a[1,0].set_ylabel('Intervals'); a[1,0].tick_params(axis='x',labelsize=6,rotation=0); a[1,0].legend(fontsize=5)
a[1,1].boxplot([S.load_act[S.action==k] for k in ac.index],tick_labels=list(ac.index))
a[1,1].set_title('(D) Load by selected action'); a[1,1].set_ylabel('kW'); a[1,1].tick_params(axis='x',labelsize=6)
save(f,'image15.png',1.376)

CF=pd.read_csv(WORK+'/pipe/counterfactuals.csv')
f,a=plt.subplots(2,2); x=np.arange(len(CF)); w=.27
a[0,0].bar(x-w,CF.Original_kW,w,label='Factual',color='#444'); a[0,0].bar(x,CF.Target_kW,w,label='Requested',color=C1); a[0,0].bar(x+w,CF.Achieved_kW,w,label='Attained',color=C3)
a[0,0].set_xticks(x); a[0,0].set_xticklabels([r.replace('-demand','') for r in CF.Record],fontsize=6)
a[0,0].legend(fontsize=6); a[0,0].set_title('(A) Demand target tracking'); a[0,0].set_ylabel('kW')
a[0,1].bar(x,CF.Achieved_kW-CF.Target_kW,color=np.where(CF.Feasible,C3,'#c53030'))
a[0,1].set_xticks(x); a[0,1].set_xticklabels([r.replace('-demand','') for r in CF.Record],fontsize=6)
a[0,1].set_title('(B) Deviation from request (red = infeasible)'); a[0,1].set_ylabel('kW')
chans=['HVAC_Actual_kW_','Chiller_Power_kW_','Humidifier_power_kW_','HV_light_Power_kW_','PV_panels_power_kW_']
lbl=['HVAC','Chiller','Humid.','Light','PV']
delta=np.array([[float(str(CF.loc[i,c]).split('->')[1])-float(str(CF.loc[i,c]).split('->')[0]) for c in chans] for i in range(len(CF))])
for k in range(len(CF)): a[1,0].bar(np.arange(len(chans))+k*.27-.27,delta[k],.27,label=CF.Record[k].replace('-demand',''))
a[1,0].set_xticks(range(len(chans))); a[1,0].set_xticklabels(lbl,fontsize=6); a[1,0].legend(fontsize=5)
a[1,0].axhline(0,color='k',lw=.8); a[1,0].set_title('(C) Recommended change by channel'); a[1,0].set_ylabel('Δ kW')
a[1,1].bar(x,100*(CF.Original_kW-CF.Achieved_kW)/CF.Original_kW,color=C2); a[1,1].axhline(25,color='k',ls='--',lw=.8,label='requested 25%')
a[1,1].set_xticks(x); a[1,1].set_xticklabels([r.replace('-demand','') for r in CF.Record],fontsize=6)
a[1,1].legend(fontsize=6); a[1,1].set_title('(D) Attained reduction'); a[1,1].set_ylabel('%')
save(f,'image16.png',1.81)
print('ALL FIGURES DONE')

## Stages 10–16 — Verification and extended analyses
*Requires Stages 1–4a in the same session.* Each stage prints progress and writes a JSON file to `{WORK}/results/`, copied to `MyDrive/Outputs/ATESI_SBEC/` when Google Drive is mounted.

In [ ]:
import os, json, shutil
G = globals()
RES = WORK + '/results'; os.makedirs(RES, exist_ok=True)
DRIVE_OUT = '/content/drive/MyDrive/Outputs/ATESI_SBEC'
def save(name, obj):
    fn = f'{RES}/{name}.json'
    json.dump(obj, open(fn, 'w'), indent=1, default=float)
    if os.path.isdir('/content/drive/MyDrive'):
        os.makedirs(DRIVE_OUT, exist_ok=True); shutil.copy(fn, DRIVE_OUT)
    print('saved', fn, flush=True)

### Stage 10 — Leakage checks
*Section 3.1.* Future-perturbation invariance, partition isolation and preprocessing audit.

In [ ]:
import numpy as np, pandas as pd, glob
df = G['df']; build = G['build']; H = 4; TGT = G['TGT']
X, y = G['X'], G['y']
out = {}

# (a) future-perturbation invariance at the feature level (on the resampled series)
rng = np.random.default_rng(0)
fails = 0; tested = 0
pos_list = rng.choice(np.arange(int(len(df) * .2), len(df) - 200), size=20, replace=False)
for pos in pos_list:
    t_star = df.index[pos]
    d2 = df.copy()
    fut = d2.index > t_star
    d2.loc[fut] = rng.normal(0, 50, size=d2.loc[fut].shape)
    G['df'] = d2
    X2, _ = build(H)
    G['df'] = df
    common = X.index[(X.index <= t_star)].intersection(X2.index)
    a = X.loc[common[-500:]]; b = X2.loc[common[-500:]]
    diff = (a - b).abs().max()
    tested += 1
    if (diff > 1e-9).any():
        fails += 1
        print('FAIL at', t_star, list(diff[diff > 1e-9].index)[:5])
out['future_perturbation'] = dict(issue_times_tested=tested, rows_per_test=500, columns=X.shape[1], failures=fails)
print(out['future_perturbation'], flush=True)

# (b) partition isolation: rebuild features with the test partition deleted
i = int(len(X) * .8); cut = X.index[i - 1]
G['df'] = df.loc[:cut]
Xt, _ = build(H)
G['df'] = df
common = Xt.index.intersection(X.index[:i])
d = (X.loc[common] - Xt.loc[common]).abs().max()
out['partition_isolation'] = dict(train_rows_compared=len(common), max_abs_diff=float(d.max()),
                                  columns_differing=int((d > 1e-9).sum()))
# rows lost at the end of train because target y(t+h) needs the future
out['partition_isolation']['note_train_rows_missing_after_rebuild'] = int(i - len(common))
print(out['partition_isolation'], flush=True)

# (c) preprocessing-level check: interpolation in Stage 1 uses r.interpolate(limit=4)
raw = G['df']  # after interpolation
RAW = sorted(glob.glob('/tmp/atesi/data/*.csv'))
r = G['load'](RAW).resample('15min').mean()
nan_rows = r.isna().any(axis=1)
filled = nan_rows & r.index.isin(raw.index)
out['interpolation'] = dict(intervals_with_missing=int(nan_rows.sum()),
                            intervals_filled_by_two_sided_linear_interpolation=int(filled.sum()),
                            intervals_dropped=int(nan_rows.sum() - filled.sum()),
                            fraction_of_kept=float(filled.sum() / len(raw)))
# how many test feature rows depend (through lags up to 672 steps) on an interpolated interval?
fill_idx = set(r.index[filled])
out['interpolation']['filled_in_test_window'] = int(sum(1 for t in r.index[filled] if t >= X.index[i]))
print(out['interpolation'], flush=True)

# (d) effect of replacing two-sided interpolation by causal forward fill on the deployed holdout score
rf = r.ffill(limit=4).dropna()
G['df'] = rf
Xf, yf = build(H); jf = int(len(Xf) * .8)
mf = G['MK'](); mf.fit(Xf.iloc[:jf], yf.iloc[:jf])
out['ffill_holdout'] = G['ev'](yf.iloc[jf:], mf.predict(Xf.iloc[jf:]))
G['df'] = df
print('ffill holdout', out['ffill_holdout'], flush=True)
save('s10_leakage', out)


### Stage 11 — ARIMA, LSTM and Transformer baselines with Diebold-Mariano tests
*Section 4.1, Tables 5 and 6.* Sequence models stop when validation error fails to improve by 0.5 % for two epochs.

In [ ]:
import numpy as np, pandas as pd, time, warnings, itertools
from scipy import stats
warnings.filterwarnings('ignore')
df = G['df']; X, y, Xtr, Xte, ytr, yte, pred = (G[k] for k in ['X', 'y', 'Xtr', 'Xte', 'ytr', 'yte', 'pred'])
ev = G['ev']; TGT = G['TGT']; H = 4
pos = pd.Series(np.arange(len(df)), index=df.index)
ptr = pos.loc[Xtr.index].values; pte = pos.loc[Xte.index].values
yser = df[TGT].values
out = {}

def dm(e1, e2, h=4, power=2):
    d = np.abs(e1) ** power - np.abs(e2) ** power
    T = len(d); dbar = d.mean(); L = h - 1
    var = np.mean((d - dbar) ** 2)
    for l in range(1, L + 1):
        var += 2 * (1 - l / (L + 1)) * np.mean((d[l:] - dbar) * (d[:-l] - dbar))
    stat = dbar / np.sqrt(var / T)
    return stat, 2 * (1 - stats.t.cdf(abs(stat), T - 1))

# ---------------- ARIMA ----------------
from statsmodels.tsa.statespace.sarimax import SARIMAX
sel = yser[ptr[-1] - 20000:ptr[-1] + 1]
best = None
for p, d, q in itertools.product([1, 2, 3], [0, 1], [0, 1]):
    t0 = time.time()
    try:
        r = SARIMAX(sel, order=(p, d, q), trend='c' if d == 0 else 'n').fit(disp=False, maxiter=200)
        print(f'ARIMA({p},{d},{q}) AIC={r.aic:.1f}  {time.time()-t0:.1f}s', flush=True)
        if best is None or r.aic < best[0]: best = (r.aic, (p, d, q), r)
    except Exception as e:
        print('ARIMA', (p, d, q), 'failed', e, flush=True)
aic, order, rsel = best
print('selected order', order, flush=True)
full = SARIMAX(yser, order=order, trend='c' if order[1] == 0 else 'n').filter(rsel.params)
fr = full.filter_results
Tm = fr.transition[:, :, 0]; Z = fr.design[:, :, 0]; c = fr.state_intercept[:, 0]; dd = fr.obs_intercept[0, 0]
A = fr.predicted_state  # a_{t|t-1}, shape (k, n+1)
# check h=1 consistency
chk = (Z @ A[:, pte[:5] + 1]).ravel() + dd
print('h=1 check', chk, full.predict(start=pte[0] + 1, end=pte[0] + 5)[:5] if hasattr(full, 'predict') else '', flush=True)
a = A[:, pte + 1]
for _ in range(H - 1): a = Tm @ a + c[:, None]
p_arima = (Z @ a).ravel() + dd
out['ARIMA'] = dict(order=order, aic=aic, **ev(yte, p_arima))
print('ARIMA', out['ARIMA'], flush=True)

# ---------------- sequence models ----------------
import torch, torch.nn as nn
torch.manual_seed(42); np.random.seed(42); torch.set_num_threads(2)
W = 48
CH = list(df.columns)
mu = np.array(df.iloc[:ptr[-1] + 1].mean().values); sd = np.array(df.iloc[:ptr[-1] + 1].std().values); sd[sd == 0] = 1
Zs = ((df.values - mu) / sd).astype(np.float32)
calcols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'moy_sin', 'moy_cos', 'is_weekend']
CAL = X[calcols].values.astype(np.float32)
ymu, ysd = ytr.mean(), ytr.std()
def mk_idx(positions, rows):
    keep = positions >= W - 1
    return positions[keep], rows[keep]
tr_pos, tr_rows = mk_idx(ptr, np.arange(len(ptr)))
te_pos, te_rows = mk_idx(pte, np.arange(len(pte)))
assert len(te_pos) == len(pte)
off = np.arange(-W + 1, 1)
def batch(positions, rows, base):
    seq = torch.from_numpy(Zs[positions[:, None] + off[None, :]])
    cal = torch.from_numpy(CAL[base + rows])
    return seq, cal
ytr_n = ((ytr.values - ymu) / ysd).astype(np.float32)

class LSTMNet(nn.Module):
    def __init__(s):
        super().__init__(); s.r = nn.LSTM(len(CH), 64, batch_first=True); s.h = nn.Sequential(nn.Linear(64 + 7, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(s, x, c):
        o, _ = s.r(x); return s.h(torch.cat([o[:, -1], c], 1)).squeeze(1)
class TrNet(nn.Module):
    def __init__(s):
        super().__init__(); s.p = nn.Linear(len(CH), 32); s.pe = nn.Parameter(torch.zeros(1, W, 32))
        s.e = nn.TransformerEncoder(nn.TransformerEncoderLayer(32, 4, 64, dropout=0.1, batch_first=True), 2)
        s.h = nn.Sequential(nn.Linear(32 + 7, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(s, x, c):
        z = s.e(s.p(x) + s.pe); return s.h(torch.cat([z[:, -1], c], 1)).squeeze(1)

def train(net, name, max_epochs=30, patience=2, min_rel=0.005, bs=1024):
    nv = int(len(tr_pos) * .1)
    fit_p, fit_r = tr_pos[:-nv], tr_rows[:-nv]; val_p, val_r = tr_pos[-nv:], tr_rows[-nv:]
    opt = torch.optim.Adam(net.parameters(), 1e-3); lossf = nn.MSELoss()
    best = np.inf; bad = 0; state = None; log = []
    for ep in range(max_epochs):
        t0 = time.time(); net.train(); perm = np.random.permutation(len(fit_p)); tl = 0
        for k in range(0, len(perm), bs):
            j = perm[k:k + bs]; s_, c_ = batch(fit_p[j], fit_r[j], 0)
            opt.zero_grad(); l = lossf(net(s_, c_), torch.from_numpy(ytr_n[fit_r[j]])); l.backward(); opt.step(); tl += l.item() * len(j)
        net.eval(); vp = predict(net, val_p, val_r, 0)
        vl = float(np.mean((vp - ytr_n[val_r]) ** 2))
        log.append((ep + 1, tl / len(perm), vl))
        print(f'{name} epoch {ep+1}: train {tl/len(perm):.4f}  val {vl:.4f}  ({time.time()-t0:.0f}s)', flush=True)
        if vl < best * (1 - min_rel): best = vl; bad = 0; state = {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: print(f'{name}: stop, val improvement < {min_rel:.1%} for {patience} epochs', flush=True); break
    net.load_state_dict(state); return log
@torch.no_grad()
def predict(net, P_, R_, base, bs=4096):
    net.eval(); o = []
    for k in range(0, len(P_), bs):
        s_, c_ = batch(P_[k:k + bs], R_[k:k + bs], base); o.append(net(s_, c_).numpy())
    return np.concatenate(o)
for name, net in [('LSTM', LSTMNet()), ('Transformer encoder', TrNet())]:
    t0 = time.time(); log = train(net, name); tt = time.time() - t0
    p_ = predict(net, te_pos, te_rows, len(Xtr)) * ysd + ymu
    out[name] = dict(window=W, epochs=len(log), train_s=tt, n_params=sum(p.numel() for p in net.parameters()), **ev(yte, p_))
    out[name + '_pred'] = None
    np.save(f'{RES}/pred_{name.split()[0]}.npy', p_)
    print(name, out[name], flush=True)
np.save(f'{RES}/pred_ARIMA.npy', p_arima)

# ---------------- DM vs LightGBM ----------------
e0 = yte.values - pred.values
dmr = {}
for n, f in [('ARIMA', f'{RES}/pred_ARIMA.npy'), ('LSTM', f'{RES}/pred_LSTM.npy'), ('Transformer', f'{RES}/pred_Transformer.npy')]:
    e = yte.values - np.load(f)
    s2, p2 = dm(e, e0, 4, 2); s1, p1 = dm(e, e0, 4, 1)
    dmr[n] = dict(DM_sq=s2, p_sq=p2, DM_abs=s1, p_abs=p1)
out['DM'] = dmr
out = {k: v for k, v in out.items() if v is not None}
print(out['DM'], flush=True)
save('s11_baselines', out)


### Stage 12 — LIME rank consistency and measured dose-response
*Section 4.4.*

In [ ]:
import numpy as np, pandas as pd, time
from sklearn.linear_model import Ridge
from scipy.stats import kendalltau, spearmanr
X, y, Xtr, Xte, ytr, yte, pred, m, MK = (G[k] for k in ['X', 'y', 'Xtr', 'Xte', 'ytr', 'yte', 'pred', 'm', 'MK'])
cols = list(Xtr.columns)
mu = Xtr.mean().values; sd = np.array(Xtr.std().values); sd[sd == 0] = 1.0

def lime(model, x0, seed, n=5000):
    rng = np.random.default_rng(seed)
    z0 = (x0 - mu) / sd
    Z = rng.normal(0, 1, size=(n, len(cols))); Z[0] = z0
    yh = model.predict(Z * sd + mu)
    d = np.sqrt(((Z - z0) ** 2).sum(1)); w = np.exp(-(d ** 2) / ((np.sqrt(len(cols)) * .75) ** 2))
    s = Ridge(alpha=1.0).fit(Z, yh, sample_weight=w)
    return s.coef_

recs = {}
for lab, q in [('Low', .10), ('Medium', .50), ('Peak', .95)]:
    i = int(np.argmin(np.abs(pred.values - pred.quantile(q)))); recs[lab] = Xte.iloc[i].values.astype(float)
    print(lab, Xte.index[i], flush=True)

# models: deployed + 10 backtest-origin refits (same scheme as Stage 4b)
models = {'deployed': m}
n = len(X); start = int(n * .5); fold = (n - start) // 10
for k in range(10):
    t0 = time.time(); mm = MK(); mm.fit(X.iloc[:start + k * fold], y.iloc[:start + k * fold]); models[f'origin{k+1}'] = mm
    print(f'refit origin {k+1}/10 ({time.time()-t0:.0f}s)', flush=True)

out = {}
TOPK = 8
for lab, x0 in recs.items():
    A = {}
    for mn, mod in models.items():
        for seed in range(10):
            A[(mn, seed)] = lime(mod, x0, seed)
        print(lab, mn, 'done', flush=True)
    keys = list(A)
    # seed consistency (deployed model, 10 seeds)
    ds = [A[('deployed', s)] for s in range(10)]
    tau_seed = [kendalltau(np.abs(ds[a]), np.abs(ds[b]))[0] for a in range(10) for b in range(a + 1, 10)]
    # refit consistency (seed 0, 11 models)
    rm = [A[(mn, 0)] for mn in models]
    tau_refit = [kendalltau(np.abs(rm[a]), np.abs(rm[b]))[0] for a in range(len(rm)) for b in range(a + 1, len(rm))]
    # rank occupancy of the deployed top-k features across all 110 explanations
    ref_top = np.argsort(-np.abs(np.mean(ds, 0)))[:TOPK]
    occ = {}
    for j in ref_top:
        ranks = [int(np.where(np.argsort(-np.abs(a)) == j)[0][0]) + 1 for a in A.values()]
        signs = [np.sign(a[j]) for a in A.values()]
        occ[cols[j]] = dict(median_rank=float(np.median(ranks)), frac_in_topk=float(np.mean(np.array(ranks) <= TOPK)),
                            sign_agreement=float(abs(np.mean(signs))), mean_coef=float(np.mean([a[j] for a in A.values()])))
    out[lab] = dict(tau_seed_mean=float(np.mean(tau_seed)), tau_seed_min=float(np.min(tau_seed)),
                    tau_refit_mean=float(np.mean(tau_refit)), tau_refit_min=float(np.min(tau_refit)), top_features=occ)
    print(lab, {k: v for k, v in out[lab].items() if k != 'top_features'}, flush=True)

# dose-response on measurements (training partition): realized target vs quantile bin of each sub-meter, within hour
SUB = [c for c in cols if c.endswith('_lag0') and not c.startswith('y_')]
hour = Xtr.index.hour
dr = {}
for c in SUB:
    rhos = []; wts = []
    for h_ in range(24):
        msk = hour == h_
        xv = Xtr[c].values[msk]; yv = ytr.values[msk]
        if np.unique(xv).size < 10: continue
        b = pd.qcut(pd.Series(xv).rank(method='first'), 5, labels=False)
        mb = pd.Series(yv).groupby(b.values).mean()
        r = spearmanr(mb.index, mb.values)[0]
        if np.isfinite(r): rhos.append(r); wts.append(msk.sum())
    rho = float(np.average(rhos, weights=wts)) if rhos else float('nan')
    frac_mono = float(np.mean(np.abs(rhos) == 1.0)) if rhos else float('nan')
    j = cols.index(c)
    lime_sign = {lab: float(np.sign(np.mean([lime(m, x0, s)[j] for s in range(10)]))) for lab, x0 in recs.items()}
    dr[c] = dict(binned_spearman_mean=rho, frac_hours_strictly_monotone=frac_mono, hours=len(rhos),
                 lime_sign=lime_sign, agrees={k: bool(np.sign(rho) == v) for k, v in lime_sign.items()})
    print(c, dr[c], flush=True)
out['dose_response'] = dr
save('s13_lime', out)


### Stage 13 — Robustness: sensor noise, missing data, tariff ratio, carbon-intensity error
*Section 4.5.*

In [ ]:
import numpy as np, pandas as pd, pulp
df, build, m, ev = G['df'], G['build'], G['m'], G['ev']
X, Xte, yte, pred = G['X'], G['Xte'], G['yte'], G['pred']
TGT = G['TGT']; H = 4; i0 = int(len(X) * .8); t_test = X.index[i0]
out = {'noise': [], 'missing': [], 'tariff': [], 'carbon': []}
SUB = [c for c in df.columns if c != TGT]
test_mask = df.index >= t_test - pd.Timedelta(days=8)  # perturb test window + lag look-back
sds = df.loc[df.index < t_test].std()

def score(d2):
    G['df'] = d2; X2, _ = build(H); G['df'] = df
    common = X2.index.intersection(Xte.index)
    return ev(yte.loc[common], m.predict(X2.loc[common])), len(common)
for lev in [0.05, 0.10, 0.20]:
    rs = []
    for seed in range(3):
        rng = np.random.default_rng(seed); d2 = df.copy()
        for c in df.columns:  # sensor noise on every metered channel incl. the net-demand meter used as input
            d2.loc[test_mask, c] = d2.loc[test_mask, c] + rng.normal(0, lev * sds[c], test_mask.sum())
        # target itself must stay clean: evaluate against true yte
        r, n = score(d2); rs.append(r['R2'])
    out['noise'].append(dict(level=lev, R2_mean=float(np.mean(rs)), R2_sd=float(np.std(rs))))
    print('noise', out['noise'][-1], flush=True)
for rate in [0.05, 0.10, 0.20]:
    rs = []
    for seed in range(3):
        rng = np.random.default_rng(100 + seed); d2 = df.copy()
        miss = test_mask & (rng.random(len(df)) < rate)
        d2.loc[miss, :] = np.nan
        d2 = d2.ffill()  # causal fill
        r, n = score(d2); rs.append(r['R2'])
    out['missing'].append(dict(rate=rate, R2_mean=float(np.mean(rs)), R2_sd=float(np.std(rs)), n=n))
    print('missing', out['missing'][-1], flush=True)
print('baseline', ev(yte, pred), flush=True)

# tariff ratio: scale (on - off) and (mid - off) by k, off fixed
TOU0 = dict(G['TOU'])
for k in [0.5, 1.0, 1.5]:
    G['TOU']['on'] = TOU0['off'] + k * (TOU0['on'] - TOU0['off']); G['TOU']['mid'] = TOU0['off'] + k * (TOU0['mid'] - TOU0['off'])
    _, su = G['run'](pred.values, yte.values, pred.index, w_carbon=0.10, ndays=30)
    out['tariff'].append(dict(k=k, on_off_ratio=G['TOU']['on'] / G['TOU']['off'], cost_saving_pct=su['cost_saving_pct'],
                              co2_saving_pct=su['co2_saving_pct'], cycles=su['cycles']))
    print('tariff', out['tariff'][-1], flush=True)
G['TOU'].update(TOU0)

# carbon-intensity uncertainty: optimiser sees the nominal profile; emissions accounted under a perturbed profile
R, su = G['run'](pred.values, yte.values, pred.index, w_carbon=0.10, ndays=30)
for sig in [0.10, 0.20, 0.30]:
    vals = []
    for seed in range(20):
        rng = np.random.default_rng(seed)
        # multiplicative lognormal error, AR(1) across intervals (rho = 0.95) to mimic persistent forecast bias
        e = np.zeros(len(R)); z = rng.normal(0, 1, len(R))
        for t in range(1, len(R)): e[t] = 0.95 * e[t - 1] + np.sqrt(1 - .95 ** 2) * z[t]
        ci_true = R.ci.values * np.exp(sig * e - sig ** 2 / 2)
        vals.append(100 * (1 - (R.grid_act * 0.25 * ci_true).sum() / (np.maximum(R.load_act, 0) * 0.25 * ci_true).sum()))
    out['carbon'].append(dict(sigma=sig, co2_saving_mean=float(np.mean(vals)), co2_saving_p05=float(np.percentile(vals, 5)),
                              co2_saving_p95=float(np.percentile(vals, 95)), nominal=su['co2_saving_pct']))
    print('carbon', out['carbon'][-1], flush=True)
save('s14_robust', out)


### Stage 14 — Reference controllers and demand-charge accounting
*Section 4.7, Table 11.*

In [ ]:
import numpy as np, pandas as pd
pred, yte = G['pred'], G['yte']; run = G['run']
CAP, P_MAX, ETA_C, ETA_D, LOSS, RAMP, DT, SOC0 = (G[k] for k in ['CAP', 'P_MAX', 'ETA_C', 'ETA_D', 'LOSS', 'RAMP', 'DT', 'SOC0'])
tou_rate, ci_f, DEM = G['tou_rate'], G['carbon_intensity'], G['DEMAND_CHARGE']
N = 29 * 96  # same 2,784 intervals as the MILP run (29 daily re-solves)
idx = pred.index[:N]; pv = pred.values[:N]; av = yte.values[:N]
rates = np.array([tou_rate(t) for t in idx]); ci = np.array([ci_f(t) for t in idx])

def account(c, d, soc, name):
    g = np.maximum(av + c - d, 0); b = np.maximum(av, 0)
    r = dict(controller=name,
             cost_base=float((b * DT * rates).sum()), cost=float((g * DT * rates).sum()),
             co2_base=float((b * DT * ci).sum()), co2=float((g * DT * ci).sum()),
             peak_base=float(b.max()), peak=float(g.max()),
             cycles=float(d.sum() * DT / CAP), soc_min=float(soc.min() / CAP), soc_max=float(soc.max() / CAP))
    r['cost_saving_pct'] = 100 * (1 - r['cost'] / r['cost_base']); r['co2_saving_pct'] = 100 * (1 - r['co2'] / r['co2_base'])
    r['peak_change_pct'] = 100 * (r['peak'] / r['peak_base'] - 1)
    # demand-charge inclusive (monthly coincident-peak charge, 30-day window)
    r['cost_incl_demand'] = r['cost'] + DEM * r['peak']; r['cost_base_incl_demand'] = r['cost_base'] + DEM * r['peak_base']
    r['cost_saving_incl_demand_pct'] = 100 * (1 - r['cost_incl_demand'] / r['cost_base_incl_demand'])
    return r

def simulate(policy, name):
    soc = SOC0; pc = pd_ = 0.0; C = []; D = []; S = []
    for t in range(N):
        want_c, want_d = policy(t, soc)
        room_c = max(0.0, (G['SOC_MAX'] * CAP - (1 - LOSS) * soc) / (ETA_C * DT)); room_d = max(0.0, ((1 - LOSS) * soc - G['SOC_MIN'] * CAP) * ETA_D / DT)
        c = min(max(want_c, pc - RAMP, 0.0), pc + RAMP, P_MAX, room_c)   # ramp-limited both ways
        d = min(max(want_d, pd_ - RAMP, 0.0), pd_ + RAMP, P_MAX, room_d)
        if c > 0 and d > 0:                                               # exclusivity: finish ramping down first
            if want_d > 0: d = 0.0
            else: c = c; d = 0.0
        soc = (1 - LOSS) * soc + ETA_C * c * DT - d * DT / ETA_D
        C.append(c); D.append(d); S.append(soc); pc, pd_ = c, d
    return account(np.array(C), np.array(D), np.array(S), name)

# conventional rule-based TES controller: charge whenever off-peak, discharge to cover load whenever on-peak, no forecast
def conv(t, soc):
    r = rates[t]
    ts = idx[t]
    if ts.dayofweek < 5 and ts.hour < 7: return P_MAX, 0.0   # weekday night off-peak block only
    if r == G['TOU']['on']: return 0.0, max(av[t - 1] if t > 0 else 0.0, 0.0)  # last measured load, no forecast
    return 0.0, 0.0
# threshold policy of Section 3.5 on the forecast (quartiles of training-window forecasts)
mtr = G['m'].predict(G['Xtr']); th1, th2 = np.quantile(mtr, .25), np.quantile(mtr, .75)
def thr(t, soc):
    if pv[t] <= th1: return P_MAX, 0.0
    if pv[t] > th2: return 0.0, max(pv[t], 0.0)
    return 0.0, 0.0
rows = [simulate(conv, 'Conventional rule-based TES controller'), simulate(thr, 'Threshold policy (Section 3.5)')]
# MILP at w=0.10 for reference, with demand-charge-inclusive accounting
R, su = run(pred.values, yte.values, pred.index, w_carbon=0.10, ndays=30)
rows.append(dict(controller='MILP w=0.10', bill_saving_pct=100 * (1 - su['bill_opt'] / su['bill_base']), cost_base=su['cost_base'], cost=su['cost_opt'], co2_base=su['co2_base'], co2=su['co2_opt'],
                 cost_saving_pct=su['cost_saving_pct'], co2_saving_pct=su['co2_saving_pct'], cycles=su['cycles'],
                 peak_base=su['peak_base'], peak=su['peak_opt'], peak_change_pct=-su['peak_reduction_pct'],
                 cost_saving_incl_demand_pct=100 * (1 - (su['cost_opt'] + DEM * su['peak_opt']) / (su['cost_base'] + DEM * su['peak_base']))))
T = pd.DataFrame(rows); print(T.round(3).to_string(), flush=True)
save('s15_controllers', dict(rows=rows, thresholds=[float(th1), float(th2)], milp_N=len(R)))


### Stage 15 — Upper-quantile demand term
*Section 4.7.*

In [ ]:
import numpy as np, pandas as pd
from lightgbm import LGBMRegressor
Xtr, ytr, Xte, yte, pred, run = G['Xtr'], G['ytr'], G['Xte'], G['yte'], G['pred'], G['run']
rows = []
for alpha in [0.90, 0.95]:
    qm = LGBMRegressor(objective='quantile', alpha=alpha, n_estimators=600, learning_rate=.05, num_leaves=63,
                       subsample=.8, colsample_bytree=.8, n_jobs=-1, random_state=42, verbose=-1)
    qm.fit(Xtr, ytr); up = qm.predict(Xte)
    cov = float(np.mean(yte.values <= up))
    print(f'quantile {alpha}: empirical coverage on held-out window {cov:.3f}', flush=True)
    R, su = run(pred.values, yte.values, pred.index, w_carbon=0.10, ndays=30, upper=up)
    su.update(alpha=alpha, coverage=cov, bill_saving_pct=100 * (1 - su['bill_opt'] / su['bill_base']))
    rows.append(su); print({k: round(v, 3) for k, v in su.items() if isinstance(v, (float, int))}, flush=True)
    if alpha == 0.95: R.to_csv('/tmp/atesi/pipe/dispatch_schedule_q95.csv')
R0, s0 = run(pred.values, yte.values, pred.index, w_carbon=0.10, ndays=30)
s0.update(alpha=None, bill_saving_pct=100 * (1 - s0['bill_opt'] / s0['bill_base'])); rows.insert(0, s0)
save('s17_quantile', rows)


### Stage 16 — Billing-period demand charge
*Section 4.7.*

In [ ]:
import numpy as np
from lightgbm import LGBMRegressor
Xtr,ytr,Xte,yte,pred,run=G['Xtr'],G['ytr'],G['Xte'],G['yte'],G['pred'],G['run']
qm=LGBMRegressor(objective='quantile',alpha=0.95,n_estimators=600,learning_rate=.05,num_leaves=63,subsample=.8,colsample_bytree=.8,n_jobs=-1,random_state=42,verbose=-1).fit(Xtr,ytr)
up=qm.predict(Xte)
rows=[]
for lab,kw in [('daily-apportioned, point',{}),('billing-period, point',dict(monthly_demand=True)),('billing-period, q95',dict(monthly_demand=True,upper=up))]:
    for w in [0.0,0.10,0.20]:
        R,su=run(pred.values,yte.values,pred.index,w_carbon=w,ndays=30,**kw)
        su.update(variant=lab,bill_saving_pct=100*(1-su['bill_opt']/su['bill_base']))
        rows.append(su)
        print(f"{lab:28s} w={w}: energy {su['cost_saving_pct']:6.2f}%  bill {su['bill_saving_pct']:6.2f}%  peak {su['peak_opt']:.2f}  CO2 {su['co2_saving_pct']:5.2f}%  cyc {su['cycles']:.2f} solve {su['mean_solve_s']:.2f}/{su['max_solve_s']:.2f}",flush=True)
        if lab.startswith('billing-period, q95') and w==0.10: R.to_csv('/tmp/atesi/pipe/dispatch_schedule_monthly_q95.csv')
save('s18_monthly',rows)


## Artefact index

Written to `{WORK}/pipe/`:

| File | Article object |
|---|---|
| `resampled.csv` | 15-minute control record, Section 3.1 |
| `forecast_h1.csv`, `forecast_h4.csv` | Tables 4 and 5 |
| `leakage_ablation.csv` | Table 7 |
| `dm_tests.csv` | Table 6 |
| `backtest.csv`, `backtest_ci.json` | Table 8 |
| `seasonwise.csv` | Table 9 |
| `stratified.csv` | Table 10 |
| `pareto.csv` | Table 11 |
| `sensitivity.csv` | Table 12 |
| `dispatch_schedule.csv`, `decision_log.csv` | Table 13 |
| `counterfactuals.csv`, `cf_stability.json` | Table 14 and Section 4.10 |
| `peakcap.csv` | Section 4.7 peak-cap experiment |

Written to `{WORK}/results/` by Stages 10–16: `s10_leakage.json`, `s11_baselines.json`,
`s13_lime.json`, `s14_robust.json`, `s15_controllers.json`, `s17_quantile.json`, `s18_monthly.json`.